# Phase 6 DAEAC FCBA RR Late-Fusion NSV + DANN: All Domain Scenarios

Keeps the FCBA + RR late-fusion NSV architecture and replaces center-based DAEAC adaptation with DANN adversarial adaptation.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd
os.environ['PYTHONUNBUFFERED'] = '1'  # stream subprocess logs/tqdm live on Kaggle

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'

SCENARIOS = [
    ('ds1_ds2', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_dann_ds1_ds2.yaml', 'daeac_fcba_rr_nsv_dann_ds1_ds2'),
    ('ds1_incart', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_dann_ds1_incart.yaml', 'daeac_fcba_rr_nsv_dann_ds1_incart'),
    ('ds1_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_dann_ds1_svdb.yaml', 'daeac_fcba_rr_nsv_dann_ds1_svdb'),
    ('mitbih_incart', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_dann_mitbih_incart.yaml', 'daeac_fcba_rr_nsv_dann_mitbih_incart'),
    ('mitbih_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_dann_mitbih_svdb.yaml', 'daeac_fcba_rr_nsv_dann_mitbih_svdb'),
]

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Repo path:', ECG)

## 1. Clone Or Locate Repo

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
os.chdir(ECG)
print(Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy NSV RR Bundle And Late-Fusion Checkpoints

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV bundle, found {candidate_dirs}'

DATA_DIR = candidate_dirs[0]
DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(DATA_DIR.glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
manifest = DATA_DIR / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

required_npz = [
    'ds1_train.npz', 'ds1_val.npz', 'mitbih_train.npz', 'mitbih_val.npz',
    'ds2_train.npz', 'ds2_val.npz', 'ds2_test.npz',
    'incart_train.npz', 'incart_val.npz', 'incart_test.npz',
    'svdb_train.npz', 'svdb_val.npz', 'svdb_test.npz',
]
for name in required_npz:
    path = DEST / name
    assert path.exists(), f'Missing {path}'
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['rr_features'].shape)

CHECKPOINTS = {
    'daeac_fcba_rr_nsv_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_sqrtw/checkpoints/daeac_fcba_rr_nsv_sqrtw_base_best.pt'),
    'daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw/checkpoints/daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt'),
}
for filename, dest in CHECKPOINTS.items():
    candidates = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    print(filename, 'candidates:', candidates)
    assert len(candidates) == 1, f'Expected exactly one {filename}, found {candidates}'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], dest)

!ls -lh data/processed/phase6_daeac_record_splits_nsv | head
!find outputs -path '*checkpoints*best.pt' -maxdepth 5 -type f -print

## 4. Validate

In [ ]:
!python scripts/check_repo.py
for scenario, config, _ in SCENARIOS:
    print('\nVALIDATE', scenario)
    subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/00_validate_data.py', '--config', config], check=True)

## 5. Smoke Run

In [ ]:
subprocess.run([
    sys.executable, 'scripts/phase6_daeac_adversarial/01_train_dann.py',
    '--config', SCENARIOS[0][1],
    '--epochs', '1',
    '--max-source-samples', '512',
    '--max-target-samples', '512',
    '--max-val-samples', '512',
    '--checkpoint-prefix', f'{SCENARIOS[0][2]}_smoke',
], check=True)

## 6. Full Adaptation Run

In [ ]:
for scenario, config, prefix in SCENARIOS:
    best = Path(f'outputs/phase6_daeac_fcba_latefusion_rr_nsv_dann_{scenario}/checkpoints/{prefix}_best.pt')
    if best.exists():
        print('resume: skip', scenario)
        continue
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_adversarial/01_train_dann.py', '--config', config], check=True)

## 7. Evaluate And Summarize

In [ ]:
rows = []
for scenario, config, prefix in SCENARIOS:
    output_dir = Path(f'outputs/phase6_daeac_fcba_latefusion_rr_nsv_dann_{scenario}')
    checkpoint = output_dir / 'checkpoints' / f'{prefix}_best.pt'
    assert checkpoint.exists(), checkpoint
    method = f'{prefix}_best'
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_adversarial/02_eval.py',
        '--config', config,
        '--checkpoint', str(checkpoint),
        '--method-name', method,
        '--dataset', 'target',
    ], check=True)
    metrics_path = output_dir / 'metrics' / f'{method}_target_test_metrics.json'
    cm_path = output_dir / 'metrics' / f'{method}_target_test_confusion_matrix.csv'
    train_summary_path = output_dir / 'metrics' / f'{prefix}_train_summary.json'
    assert metrics_path.exists(), metrics_path
    metrics = json.loads(metrics_path.read_text())
    train_summary = json.loads(train_summary_path.read_text()) if train_summary_path.exists() else {}
    rows.append({
        'scenario': scenario,
        'best_epoch': train_summary.get('best_epoch'),
        'best_v_measure': train_summary.get('best_v_measure'),
        'target_accuracy': metrics['accuracy'],
        'target_macro_f1': metrics['macro_f1'],
        'N_f1': metrics['per_class']['N']['f1'],
        'S_f1': metrics['per_class']['S']['f1'],
        'V_f1': metrics['per_class']['V']['f1'],
        'best_checkpoint': str(checkpoint),
        'metrics_path': str(metrics_path),
        'confusion_matrix_path': str(cm_path),
    })

df = pd.DataFrame(rows).sort_values('scenario')
display(df)
summary_csv = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_dann_all_domains_summary.csv')
summary_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_csv, index=False)
print('Saved', summary_csv)

## 8. Save Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_dann_all_domains_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv_dann_* outputs/phase6_daeac_fcba_latefusion_rr_nsv_dann_all_domains_summary.csv